# Lineare Regression mit Statsmodels

In den ersten Notebooks dieses Repositorys haben wir mit der Bibliothek `scikit-learn` gearbeitet, um lineare Regressionsmodelle zu trainieren. In diesem Notebook möchten wir eine weitere Bibliothek vorstellen, die für diesen Zweck verwendet werden kann: `statsmodels`.

Die Frage, welche Bibliothek gewählt werden soll, ist eng mit Ihrem Ziel verbunden. Hier sind einige der Vorteile und Anwendungsbereiche für beide Bibliotheken:

**scikit-learn:**

* bietet eine breite Palette von Modellen und Algorithmen nicht nur für Regression, sondern auch für Klassifikation, Dimensionsreduktion, Clustering usw.
* saubere und leicht zu erlernende Syntax
* Schwerpunkt auf prädiktiver Modellierung statt auf deskriptiver Statistik
* bietet Werkzeuge für Transformationspipelines

**statsmodels:**

* vielseitiges Modul für komplexe Statistiken
* bietet leistungsstarke Statistik- und Ökonometrie-Tools
* gut zum Anpassen statistischer Modelle, Durchführen statistischer Tests oder zur Datenexploration
* Schwerpunkt auf dem Verständnis relevanter Variablen und ihrer Effektgröße
* nützlich für Zeitreihen

Wie Sie sehen, haben beide Bibliotheken ihre Vorteile. Kurz gesagt, wenn Sie Ihre Daten analysieren möchten (z.B. die statistische Signifikanz verschiedener Variablen für Ihr Modell überprüfen), ist statsmodels die richtige Wahl. Wenn Ihr Fokus auf der Anpassung eines guten Modells und der Erstellung von Vorhersagen liegt, ist scikit-learn die bessere Wahl.

## Lernziele

In diesem Notebook lernen Sie…

* wie man statsmodels zum Trainieren eines linearen Regressionsmodells verwendet.
* wie man die wichtigsten Metriken der Modellzusammenfassung interpretiert.

## Der Autodatensatz

Wir bleiben bei dem zu diesem Zeitpunkt hoffentlich gut bekannten Autodatensatz, um eine einfache und eine multiple lineare Regression mit statsmodels zu trainieren.

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# Using the same car dataset as before
cars = pd.read_csv("data/cars_multivariate.csv")
cars.head()

## Einfache Lineare Regression

Für unsere einfache lineare Regression werden wir unsere Zielvariable `mpg` mit dem Feature `horsepower` modellieren.

In [ ]:
# Plot relationship between horsepower and mpg 
cars.plot('horsepower', 'mpg', kind='scatter');

Wie üblich müssen wir unsere Daten vorbereiten, indem wir das Ziel und die Feature(s) definieren. Im Gegensatz zu `scikit-learn` fügt `statsmodels` standardmäßig keinen konstanten Term zur Berechnung des Achsenabschnitts hinzu. Wenn Sie die Dokumentation für [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) überprüfen, werden Sie sehen, dass einer seiner Parameter `fit_intercept` heißt und sein Standardwert `True` ist. Wenn wir [statsmodels OLS](https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html) verwenden, müssen wir manuell einen konstanten Term hinzufügen, wenn wir möchten, dass unser Modell mit einem Achsenabschnitt angepasst wird.

In [ ]:
# Define target and feature and add constant term
X = cars.horsepower
X = sm.add_constant(X)
y = cars.mpg

X.head()

In [ ]:
# Fit model and print summary
model = sm.OLS(y, X)
results = model.fit()

# Print intercept and coefficients
results.params

Wie Sie sehen können, ist die Syntax etwas anders, aber sie liefert die gleichen Ergebnisse. Eine Funktion, die `statsmodel` unglaublich nützlich für statistische Analysen macht, ist die `.summary`, die Sie für jedes Modell ausgeben können. Sie gibt Ihnen einen Überblick über (fast) alle vorstellbaren Metriken und statistischen Zahlen, die Sie benötigen könnten. Am Anfang kann das ziemlich überwältigend sein.

In [ ]:
# Print extensive summary
results.summary()

### Modellinterpretation

Wir werden nur kurz über diese Zusammenfassung sprechen. Für eine ausführlichere Erklärung aller Metriken innerhalb der Modellzusammenfassung schauen Sie am Ende des Notebooks nach.

<center><img src="images/ols_summary.png" height="400"/></center>

* Der blaue Teil der Tabelle gibt einige Details zu den Daten und dem Modell selbst.
* Der orange Teil liefert Informationen über die Güte der Anpassung.
    * **$R^2$**
    * **adjustiertes $R^2$**
* Der grüne Teil gibt Informationen über die geschätzten Koeffizienten unseres Modells.
    * **coef** sind die Werte für den Achsenabschnitt, der der Koeffizient für die Konstante ist, sowie Koeffizienten für jedes Feature
    * **P>|t|** ist eine der wichtigsten Statistiken in der Zusammenfassung. Sie verwendet die t-Statistik, um den p-Wert zu erzeugen, ein Maß dafür, wie wahrscheinlich Ihr Koeffizient durch unser Modell durch Zufall gemessen wird.
* Der türkisfarbene Teil unten liefert Informationen über die Residuen, Autokorrelation und Multikollinearität.

## Multiple Lineare Regression

Auch mit `statsmodel` können wir eine multiple lineare Regression mit mehreren Features trainieren.

In [ ]:
# Define target and features and add a constant term 
X2 = cars[['horsepower', 'weight']]
X2 = sm.add_constant(X2)
y2 = cars.mpg

X2.head(n=2)

In [ ]:
# We can also combine all steps using dot notation
sm.OLS(y2, X2).fit().summary()

### Vorhersagen für neue Beobachtungen

Wie `scikit-learn` hat auch `statsmodels` eine `.predict` Methode zur Berechnung von Werten für neue Instanzen. Nehmen wir an, wir möchten das `mpg` für ein neues Auto mit `horsepower` von 250 und einem `weight` von 3000 vorhersagen.
Wir können unsere neuen Daten als Liste eingeben, müssen aber sicherstellen, dass wir eine Konstante von 1 für die Achsenabschnittberechnung hinzufügen.

In [ ]:
# Make predictions on test car
test_car = [1, 250, 3000]
sm.OLS(y2, X2).fit().predict(test_car)

Laut unserem Modell hätte ein Auto mit den oben genannten Eigenschaften eine Kraftstoffeffizienz von etwa 15,9 mpg.

## Die Formelnotation

Neben der Syntax, die wir oben verwendet haben, können wir auch die sogenannte **Formelnotation** wählen. Sie liefert die gleichen Ergebnisse, nur der Code ist anders, er ähnelt leicht der R-Syntax.

In [ ]:
# mpg explained by horsepower and weight
smf.ols(formula='mpg ~ horsepower + weight', data=cars).fit().summary()

## Modellzusammenfassung - Klar erklärt...

Hier ist eine kurze Beschreibung der Zusammenfassungstabelle. Keine Sorge, Sie müssen sie jetzt nicht alle kennen oder verstehen.

#### Der blaue Teil der Tabelle gibt einige Details zu den Daten und dem Modell:

* **Dep. Variable**: Abhängige Variable (Ziel)
* **Model**: Verwendete Technik, eine abgekürzte Version der Methode
* **Method**: Die Verlustfunktion, die im Parameterauswahlprozess optimiert wird. Kleinste Quadrate, da sie die Parameter auswählt, die den Trainingsfehler reduzieren. Dies wird auch als Mittlerer Quadratischer Fehler [MSE] bezeichnet.
* **No. Observations**: Anzahl der zur Modellschulung verwendeten Beobachtungen / Größe der Trainingsdaten
* **Df Residuals**: Freiheitsgrade der Residuen, also die Anzahl der Beobachtungen - Anzahl der Parameter. Der Achsenabschnitt ist ein Parameter. Der Zweck der Freiheitsgrade besteht darin, die Auswirkungen beschreibender/zusammenfassender Statistiken im Modell widerzuspiegeln, was bei der Regression der Koeffizient ist. Da die Beobachtungen diesen Parametern "gerecht werden" müssen, haben sie nur so viele freie Beobachtungen, und der Rest muss reserviert werden, um der "Prophezeiung" der Parameter "gerecht zu werden". Dieser interne Mechanismus stellt sicher, dass genügend Beobachtungen vorhanden sind, um den Parametern zu entsprechen.
* **Df Model**: Freiheitsgrade des Modells nummerieren unsere prädiktiven Variablen. Die Anzahl der Parameter im Modell (ohne den konstanten/Achsenabschnitt-Term, falls vorhanden)
* **Covariance Type**: Robuste Regressionsmethoden sind darauf ausgelegt, nicht übermäßig von Verletzungen der Annahmen durch den zugrunde liegenden Datengenerierungsprozess beeinflusst zu werden. Da dieses Modell Ordinary Least Squares ist, ist es nicht robust und daher sehr empfindlich gegenüber Ausreißern.

#### Der orange Teil der Tabelle zeigt die Güte der Anpassung:

* **R-squared**: Das Bestimmtheitsmaß. Es quantifiziert den Prozentsatz der Varianz in der Zielvariablen, der durch das Modell erklärt wird. Der verbleibende Prozentsatz repräsentiert die durch Fehler erklärte Varianz, den E-Term, den Teil, den Modell und Prädiktoren nicht erfassen können.
* **Adj. R-squared**: Version des R-Squared, die zusätzliche unabhängige Variablen bestraft.
* **F-statistic**: Ein Maß dafür, wie signifikant die Anpassung ist. Der mittlere quadratische Fehler des Modells geteilt durch den mittleren quadratischen Fehler der Residuen. Fließt in die Berechnung des P-Werts ein.
* **Prob (F-statistic) or P-Value**: Die Wahrscheinlichkeit, dass eine Stichprobe wie diese die obige Statistik liefern würde, und ob das Urteil des Modells zur Nullhypothese konsistent die Population repräsentiert. Misst nicht die Effektgröße, sondern die Integrität und Konsistenz dieses Tests bei dieser Datengruppe.
* **Log-likelihood**:
* **AIC**: Das Akaike-Informationskriterium. Passt die Log-Likelihood basierend auf der Anzahl der Beobachtungen und der Komplexität des Modells an. Bestraft die Modellauswahlmetriken, wenn mehr unabhängige Variablen hinzugefügt werden.
* **BIC**: Das Bayesianische Informationskriterium. Ähnlich wie das AIC, hat aber eine höhere Strafe für Modelle mit mehr Parametern. Bestraft die Modellauswahlmetriken, wenn mehr unabhängige Variablen hinzugefügt werden.

#### Die grüne zweite Tabelle zeigt den Koeffizientenbericht:

* **coef**: Der geschätzte Wert des Koeffizienten. Um wie viel das Modell den unabhängigen Wert multipliziert.
* **std err**: Der grundlegende Standardfehler der Schätzung des Koeffizienten. Durchschnittliche Distanzabweichung der Punkte vom Modell, die eine einheitsrelevante Möglichkeit bietet, die Modellgenauigkeit zu messen.
* **t**: Der t-Statistikwert. Dies ist ein Maß dafür, wie statistisch signifikant der Koeffizient ist.
* **P > |t|**: P-Wert, dass die Nullhypothese, dass der Koeffizient = 0 ist, wahr ist. Wenn er kleiner als das Konfidenzniveau ist, oft 0,05, zeigt dies an, dass es eine statistisch signifikante Beziehung zwischen dem Term und der Antwort gibt.
* **[95.0% Conf. Interval]**: Die unteren und oberen Werte des 95%-Konfidenzintervalls. Spezifischer Bereich der möglichen Koeffizientenwerte.

#### Die türkisfarbene dritte Tabelle liefert Informationen über Residuen, Autokorrelation und Multikollinearität:

* **Skewness**: Ein Maß für die Symmetrie der Daten um den Mittelwert. Normalverteilte Fehler sollten symmetrisch um den Mittelwert verteilt sein (gleiche Mengen über und unter der Linie). Die Normalverteilung hat eine Schiefe von 0.
* **Kurtosis**: Ein Maß für die Form der Verteilung. Vergleicht die Datenmenge nahe dem Mittelwert mit denen weit weg vom Mittelwert (in den Rändern), also die "Spitzigkeit" des Modells. Die Normalverteilung hat eine Kurtosis von 3, und je größer die Zahl, desto mehr spitzt sich die Kurve zu.
* **Omnibus D'Angostino's test**: Es bietet einen kombinierten statistischen Test für das Vorhandensein von Schiefe und Kurtosis.
* **Prob(Omnibus)**: Die obige Statistik in eine Wahrscheinlichkeit umgewandelt
* **Jarque-Bera**: Ein anderer Test der Schiefe und Kurtosis
* **Prob (JB)**: Die obige Statistik in eine Wahrscheinlichkeit umgewandelt
* **Durbin-Watson**: Ein Test auf das Vorhandensein von Autokorrelation (dass die Fehler nicht unabhängig sind), was oft bei Zeitreihenanalysen wichtig ist. Es ist auch ein Maß für Homoskedastizität. Ein perfekter Wert läge zwischen 1 und 2.
* **Cond. No**: Konditionszahl; Ein Test auf Multikollinearität (wenn in einer Anpassung mit mehreren Parametern die Parameter miteinander in Beziehung stehen). Multikollinearität wird stark durch eine hohe Konditionszahl impliziert.